In [ ]:
# ==============================================================================
# 📓 Advanced RAG Cookbook: 05_query_transformation.ipynb
# ==============================================================================
# الهدف: تطبيق تقنيات معالجة وتطوير أسئلة المستخدم (Query Transformation) لكود إنتاجي:
# 1. Query Rewriting (إعادة الصياغة باستخدام LCEL)
# 2. Multi-Query Generation (توليد صياغات بـ Pydantic Structured Output)
# 3. Sub-Query Decomposition (تفكيك الأسئلة بـ JSON Output Parser)
# 4. Hypothetical Document Embeddings - HyDE (البحث بالـ Hypothetical Generation)
# ==============================================================================

# !pip install langchain-community langchain-core langchain-openai langchain-qdrant sentence-transformers pydantic

import os
from typing import List
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Qdrant
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI  # يمكن استبدالها بـ ChatOllama أو ChatGroq

print("✅ تم استيراد المكتبات بنجاح!")

<div dir="rtl">

## 1. إعداد الـ LLM والـ Vector Store

سنقوم بتعريف نموذج الـ LLM وقاعدة البيانات المتجهة للتطبيق عليها عبر كل الأنماط.

</div>

In [ ]:
# 1. إعداد نموذج الـ LLM (يمكنك تغير النموذج أو استخدام Ollama محلياً)
# os.environ["OPENAI_API_KEY"] = "your-api-key"
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. إعداد Vector DB تجريبي للاستعلامات
documents = [
    Document(page_content="Annual leave entitlement is 21 business days per year after completing 6 months of probation."),
    Document(page_content="Health insurance covers outpatient care up to $3,000 and inpatient care up to $20,000 annually."),
    Document(page_content="To request hardware upgrades, submit a ticket on the IT Helpdesk with your line manager's approval."),
    Document(page_content="Remote work policy allows 2 days per week from home, subject to department leads coordination.")
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = Qdrant.from_documents(documents, embeddings, location=":memory:")
retriever = vector_db.as_retriever(search_kwargs={"k": 2})

print("✅ Vector DB و LLM جاهزان لعملية الـ Transformation!")

<div dir="rtl">

## 2. التقنية الأولى: Query Rewriting (إعادة الصياغة باستخدام LCEL)

تحويل سؤال المستخدم المختصر أو المبهم إلى سؤال واضح، قائم بذاته، وغني بالمصطلحات المناسبة للـ Vector Search.

</div>

In [ ]:
# ==============================================================================
# 1. Query Rewriting Chain Implementation
# ==============================================================================

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI query optimizer. Your job is to rewrite the input user query "
               "to be clear, standalone, and optimized for vector similarity search. "
               "Do NOT answer the question. Only output the reformulated query."),
    ("human", "Original Query: {query}")
])

# بناء الـ LCEL Chain الحقيقية
rewrite_chain = rewrite_prompt | llm | StrOutputParser()

# الربط المباشر بين الـ Query Transformation والـ Retriever
def rewrite_and_retrieve(raw_query: str):
    optimized_query = rewrite_chain.invoke({"query": raw_query})
    print(f"🔹 Raw Query: {raw_query}")
    print(f"✨ Optimized Query: {optimized_query}\n")
    
    docs = retriever.invoke(optimized_query)
    return docs

# تجربة التشغيل:
# retrieved_docs = rewrite_and_retrieve("Vacation days?")

<div dir="rtl">

## 3. التقنية الثانية: Multi-Query Generation (Pydantic Structured Output)

توليد 3-5 صياغات مختلفة للبحث الموازي (Parallel Retrieval) وإزالة القطع المكررة (Deduplication).

</div>

In [ ]:
# ==============================================================================
# 2. Multi-Query Generation via Pydantic
# ==============================================================================

class MultiQuerySchema(BaseModel):
    queries: List[str] = Field(
        description="List of 3 to 5 distinct rephrasings of the original query for semantic search."
    )

# إجبار الـ LLM على الإرجاع بتنسيق Pydantic Schema
structured_llm = llm.with_structured_output(MultiQuerySchema)

multi_query_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI assistant. Generate 3 to 5 different versions or perspectives of the given "
               "user query to retrieve relevant documents from a vector database."),
    ("human", "Original Query: {query}")
])

multi_query_chain = multi_query_prompt | structured_llm

def multi_query_retrieval_pipeline(query: str):
    # 1. توليد الأسئلة الموازية بـ LLM
    response: MultiQuerySchema = multi_query_chain.invoke({"query": query})
    generated_queries = response.queries
    generated_queries.append(query) # إضافة السؤال الأصلي
    
    print(f"🎯 Generated Queries: {generated_queries}\n")
    
    # 2. الاستعلام الموازي وتجميع النتائج بدون تكرار (Deduplication)
    unique_docs = {}
    for q in generated_queries:
        docs = retriever.invoke(q)
        for doc in docs:
            unique_docs[doc.page_content] = doc
            
    return list(unique_docs.values())

# تجربة التشغيل:
# unique_results = multi_query_retrieval_pipeline("Can I work from home?")

<div dir="rtl">

## 4. التقنية الثالثة: Sub-Query Decomposition (تفكيك الأسئلة المعقدة)

تفكيك الأسئلة المركبة التي تطلب عدة معلومات معاً إلى أسئلة فرعية بسيطة والاستعلام عن كل منها بشكل منفصل.

</div>

In [ ]:
# ==============================================================================
# 3. Sub-Query Decomposition Pipeline
# ==============================================================================

class SubQueriesSchema(BaseModel):
    sub_queries: List[str] = Field(
        description="Break down the complex user question into independent, simple sub-questions."
    )

decomposition_prompt = ChatPromptTemplate.from_messages([
    ("system", "Decompose the following complex user query into smaller, independent sub-queries. "
               "Each sub-query should focus on a single specific detail."),
    ("human", "Complex Query: {query}")
])

decomposition_chain = decomposition_prompt | llm.with_structured_output(SubQueriesSchema)

def decomposed_retrieval_pipeline(complex_query: str):
    # تفكيك السؤال بواسطة الـ LLM
    sub_queries_obj: SubQueriesSchema = decomposition_chain.invoke({"query": complex_query})
    sub_queries = sub_queries_obj.sub_queries
    
    print(f"🧩 Sub-Queries: {sub_queries}\n")
    
    # الاستعلام لكل سؤال فرعي بحد ذاته
    retrieved_context = {}
    for sub_q in sub_queries:
        docs = retriever.invoke(sub_q)
        retrieved_context[sub_q] = [d.page_content for d in docs]
        
    return retrieved_context

# تجربة التشغيل:
# sub_results = decomposed_retrieval_pipeline("What are the rules for annual leave and what is the health insurance coverage limit?")

<div dir="rtl">

## 5. التقنية الرابعة: Hypothetical Document Embeddings (HyDE)

توليد إجابة/وثيقة فرضية بواسطة الـ LLM، ثم تحويل الإجابة الفرضية إلى Vector واستخدامها للبحث داخل الـ Vector DB بدلاً من متجه السؤال الأصلي.

</div>

In [ ]:
# ==============================================================================
# 4. HyDE (Hypothetical Document Embeddings) Implementation
# ==============================================================================

hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "Please write a detailed hypothetical passage or paragraph that answers the user's question. "
               "Do NOT worry about exact facts; write a realistic domain-specific paragraph."),
    ("human", "Question: {query}")
])

# الـ Chain المخصصة لإنشاء الوثيقة الفرضية
hyde_chain = hyde_prompt | llm | StrOutputParser()

def hyde_retrieval_pipeline(query: str):
    # 1. توليد المستند الفرضي بـ LLM
    hypothetical_doc = hyde_chain.invoke({"query": query})
    print(f"❓ Question: {query}")
    print(f"📝 Generated Hypothetical Document (HyDE):\n{hypothetical_doc}\n")
    
    # 2. الاستعلام في Vector DB باستخدام الـ Vector الخاص بالـ Hypothetical Document
    docs = retriever.invoke(hypothetical_doc)
    return docs

# تجربة التشغيل:
# hyde_results = hyde_retrieval_pipeline("How many paid days off do I get?")

<div dir="rtl">

## 📝 ملخص الهندسة والمعمارية (Architecture Summary)

| التقنية | متى نستخدمها؟ | النمط البرمجي المستعمل |
|---|---|---|
| **Query Rewriting** | الأسئلة القصيرة والمبهمة | `ChatPromptTemplate \| LLM \| StrOutputParser` |
| **Multi-Query** | لتغطية المصطلحات المختلفة في المستندات | `LLM.with_structured_output(PydanticSchema)` |
| **Decomposition** | الأسئلة المركبة من عدة أجزاء | `LLM.with_structured_output` + Loop Retrieval |
| **HyDE** | لما تكون الإجابات طوال والسؤال قصير | Hypothetical Generation -> Vector Search |

</div>